# C3: Proposal vs Reality

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Track unit changes** from proposal to completion
2. **Understand "penciling"** - developer financial feasibility
3. **Calculate RHNA progress** for state reporting
4. **Identify patterns** in project modifications

## Why This Matters

Proposed housing often changes during the approval process:
- Units get reduced to meet neighborhood opposition
- Affordable requirements increase costs
- Market changes affect feasibility

The Terner Center's "Making It Pencil" methodology examines why projects shrink or fail.

## RHNA Context

California's **Regional Housing Needs Allocation (RHNA)** requires cities to plan for housing at all income levels:

| Income Level | % of AMI | Berkeley 2023-2031 |
|--------------|----------|-------------------|
| Very Low (VLI) | 0-50% | 2,446 units |
| Low (LI) | 50-80% | 1,408 units |
| Moderate (MOD) | 80-120% | 1,416 units |
| Above Moderate | 120%+ | 3,664 units |

This notebook tracks progress toward these targets.

---

## Overview

Compare proposed units vs actual outcomes using "Making It Pencil" methodology.

**Metrics:**
- Proposed vs completed units
- RHNA progress by income category
- Project modification patterns

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Data

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    
    # Check for new_units vs old_units columns
    unit_cols = [c for c in df.columns if 'unit' in c.lower()]
    print(f"\nUnit-related columns: {unit_cols}")
    
    if 'new_units' in df.columns and 'old_units' in df.columns:
        print(f"\nNew units (proposed): {df['new_units'].sum():,.0f}")
        print(f"Old units (demolished): {df['old_units'].sum():,.0f}")
        print(f"Net units: {df['net_units'].sum():,.0f}")

## 3. New vs Demolished Units

In [ ]:
# Analyze new vs demolished units
if df is not None:
    # Projects with demolitions
    if 'old_units' in df.columns:
        with_demolition = df[df['old_units'] > 0]
        
        print(f"Projects with demolitions: {len(with_demolition)}")
        print(f"Total units demolished: {with_demolition['old_units'].sum():,.0f}")
        print(f"Total new units in those projects: {with_demolition['new_units'].sum():,.0f}")
        
        print("\nLargest demolitions:")
        display(with_demolition.nlargest(10, 'old_units')[['address_display', 'old_units', 'new_units', 'net_units']])

## 4. Pipeline Attrition Analysis

In [ ]:
# Analyze attrition (projects that may not complete)
if df is not None:
    # Define at-risk statuses
    at_risk_keywords = ['Incomplete', 'Corrections', 'Appealed', 'Stalled']
    
    at_risk = df[df['status'].str.contains('|'.join(at_risk_keywords), case=False, na=False)]
    
    print(f"At-risk projects: {len(at_risk)} ({100*len(at_risk)/len(df):.1f}%)")
    print(f"At-risk units: {at_risk['net_units'].sum():,.0f} ({100*at_risk['net_units'].sum()/df['net_units'].sum():.1f}%)")
    
    print("\nAt-risk by status:")
    at_risk_summary = at_risk.groupby('status').agg({
        'address_display': 'count',
        'net_units': 'sum'
    }).sort_values('net_units', ascending=False)
    at_risk_summary.columns = ['Projects', 'Units']
    display(at_risk_summary)

## 5. Success Rate by Size Category

In [ ]:
# Success rate by size
if df is not None and 'project_size_category' in df.columns:
    def get_success_indicator(status):
        status = str(status).lower()
        if 'complete' in status or 'approved' in status or 'final' in status:
            return 'Likely Success'
        elif 'review' in status:
            return 'In Progress'
        else:
            return 'At Risk'
    
    df['success_indicator'] = df['status'].apply(get_success_indicator)
    
    # Pivot by size and success
    success_by_size = pd.crosstab(
        df['project_size_category'],
        df['success_indicator'],
        values=df['net_units'],
        aggfunc='sum'
    ).fillna(0).astype(int)
    
    print("Units by Size Category and Success Indicator:")
    display(success_by_size)

## 6. Yearly Attrition Trends

In [ ]:
# Attrition by year
if df is not None and 'year' in df.columns:
    yearly_attrition = df.groupby(['year', 'success_indicator'])['net_units'].sum().unstack(fill_value=0)
    
    print("Units by Year and Success Status:")
    display(yearly_attrition)

## 7. Summary Statistics

In [ ]:
# Overall summary
if df is not None:
    total_units = df['net_units'].sum()
    
    # By success indicator
    success_summary = df.groupby('success_indicator').agg({
        'address_display': 'count',
        'net_units': 'sum'
    })
    success_summary.columns = ['Projects', 'Units']
    success_summary['% of Total'] = (success_summary['Units'] / total_units * 100).round(1)
    
    print("Proposal to Reality Summary:")
    print("="*60)
    display(success_summary)
    
    print(f"\nTotal proposed units: {total_units:,.0f}")
    
    likely_success = success_summary.loc['Likely Success', 'Units'] if 'Likely Success' in success_summary.index else 0
    print(f"Likely to complete: {likely_success:,.0f} ({likely_success/total_units*100:.1f}%)")

## 8. Export Analysis

In [ ]:
# Export proposal vs reality analysis
if df is not None:
    output_path = DATA_DIR / 'proposal_vs_reality.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook analyzed:
- New units vs demolished units
- Pipeline attrition rates
- Success rates by project size
- Yearly trends

**Next:** Run `D1_monthly_report_generator.ipynb` for automated reporting.